# SoundsLike: Hear Your Probability Distributions

This notebook demonstrates how to **sonify** probability distributions - converting statistical concepts into sound.

We're used to *seeing* probability distributions as curves and histograms. But what do they *sound* like?

Each distribution creates a unique sonic texture based on how its values are spread across frequencies.

In [ ]:
from soundslike import ProbabilitySounds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress logging for cleaner notebook output
import logging
logging.getLogger('soundslike.soundslike').setLevel(logging.WARNING)

# Create our sonifier
ps = ProbabilitySounds()
print("Ready to make some noise!")

## 1. The Normal Distribution

The normal (Gaussian) distribution is the most famous probability distribution. It's defined by:
- **Mean (μ)**: The center frequency
- **Standard deviation (σ)**: How spread out the frequencies are

Let's hear how different standard deviations affect the sound. Both examples are centered on A440 (the A above middle C).

In [ ]:
# Tight distribution (small std) - should sound focused and clear
print("Normal Distribution: μ=440Hz, σ=10Hz (tight, focused)")
ps.sonify_normal(mean=440, std=10, num_samples=50).to_audio()

In [ ]:
# Wide distribution (large std) - should sound diffuse and shimmering
print("Normal Distribution: μ=440Hz, σ=100Hz (wide, diffuse)")
ps.sonify_normal(mean=440, std=100, num_samples=50).to_audio()

### Visualizing the difference

The tight distribution concentrates frequencies around 440Hz, creating a clearer pitch.
The wide distribution spreads frequencies out, creating a more complex, "shimmery" sound.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Tight distribution
tight = np.random.normal(440, 10, 1000)
sns.histplot(tight, bins=30, ax=axes[0], kde=True)
axes[0].set_title('Tight: σ=10Hz')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_xlim(200, 680)

# Wide distribution  
wide = np.random.normal(440, 100, 1000)
sns.histplot(wide, bins=30, ax=axes[1], kde=True)
axes[1].set_title('Wide: σ=100Hz')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(200, 680)

plt.tight_layout()
plt.show()

## 2. The Uniform Distribution

The uniform distribution has equal probability across its entire range. This creates a "white noise" like sound - all frequencies are equally represented.

Compare this to the normal distribution which emphasizes the center.

In [ ]:
# Uniform distribution - equal probability across range
print("Uniform Distribution: 300-580Hz (flat, chaotic)")
ps.sonify_uniform(low=300, high=580, num_samples=50).to_audio()

In [ ]:
# Compare with normal distribution over similar range
print("Normal Distribution: μ=440Hz, σ=50Hz (same range, but centered)")
ps.sonify_normal(mean=440, std=50, num_samples=50).to_audio()

## 3. The Beta Distribution

The beta distribution is incredibly flexible. Its shape depends on two parameters: α (alpha) and β (beta).

- **α = β = 1**: Uniform (flat)
- **α = β > 1**: Bell-shaped, symmetric
- **α > β**: Skewed right (higher frequencies)
- **α < β**: Skewed left (lower frequencies)

In [ ]:
# Symmetric beta (bell-shaped)
print("Beta Distribution: α=5, β=5 (symmetric, bell-shaped)")
ps.sonify_beta(a=5, b=5, freq_range=(220, 880), num_samples=50).to_audio()

In [ ]:
# Right-skewed beta (emphasizes higher frequencies)
print("Beta Distribution: α=5, β=2 (skewed toward higher frequencies)")
ps.sonify_beta(a=5, b=2, freq_range=(220, 880), num_samples=50).to_audio()

In [ ]:
# Left-skewed beta (emphasizes lower frequencies)
print("Beta Distribution: α=2, β=5 (skewed toward lower frequencies)")
ps.sonify_beta(a=2, b=5, freq_range=(220, 880), num_samples=50).to_audio()

### Visualizing Beta Distribution Shapes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

freq_range = (220, 880)
for ax, (a, b), title in zip(axes, [(5, 5), (5, 2), (2, 5)], 
                               ['Symmetric', 'Right-skewed', 'Left-skewed']):
    samples = np.random.beta(a, b, 1000)
    samples = samples * (freq_range[1] - freq_range[0]) + freq_range[0]
    sns.histplot(samples, bins=30, ax=ax, kde=True)
    ax.set_title(f'{title}: α={a}, β={b}')
    ax.set_xlabel('Frequency (Hz)')

plt.tight_layout()
plt.show()

## 4. The Exponential Distribution

The exponential distribution creates sounds that are concentrated at lower frequencies with a long tail extending to higher frequencies. It's characterized by rapid decay - most values cluster near the minimum.

In [ ]:
print("Exponential Distribution: base=200Hz, scale=100")
ps.sonify_exponential(scale=100, base_freq=200, num_samples=50).to_audio()

In [ ]:
# Visualize
samples = np.random.exponential(100, 1000) + 200
plt.figure(figsize=(8, 4))
sns.histplot(samples, bins=30, kde=True)
plt.title('Exponential Distribution (base=200Hz, scale=100)')
plt.xlabel('Frequency (Hz)')
plt.show()

## 5. Sample Size: From Discrete to Continuous

An interesting effect: as we increase the number of samples, the sound becomes richer and more complex. With few samples, you can hear individual tones. With many samples, they blend into a texture.

In [ ]:
print("10 samples - you can hear individual tones")
ps.sonify_normal(mean=440, std=30, num_samples=10).to_audio()

In [ ]:
print("50 samples - starting to blend")
ps.sonify_normal(mean=440, std=30, num_samples=50).to_audio()

In [ ]:
print("200 samples - rich, full texture")
ps.sonify_normal(mean=440, std=30, num_samples=200).to_audio()

## 6. Custom Distributions

You can sonify any array of frequencies using the `sonify()` method directly. This lets you create sounds from any data!

In [ ]:
# Create a bimodal distribution (two peaks)
bimodal = np.concatenate([
    np.random.normal(330, 15, 25),  # E4
    np.random.normal(440, 15, 25),  # A4
])

print("Bimodal distribution (two frequency peaks: E4 and A4)")
ps.sonify(bimodal, duration=1.5).to_audio()

In [ ]:
# Visualize the bimodal distribution
plt.figure(figsize=(8, 4))
sns.histplot(bimodal, bins=20, kde=True)
plt.axvline(330, color='r', linestyle='--', label='E4 (330Hz)')
plt.axvline(440, color='g', linestyle='--', label='A4 (440Hz)')
plt.title('Bimodal Distribution')
plt.xlabel('Frequency (Hz)')
plt.legend()
plt.show()

## 7. Musical Intervals as Distributions

We can create distributions centered on musical intervals. Here's a major chord (root, major third, fifth):

In [ ]:
# A major chord: A4 (440), C#5 (554), E5 (659)
chord = np.concatenate([
    np.random.normal(440, 5, 20),   # Root (A)
    np.random.normal(554, 5, 20),   # Major third (C#)
    np.random.normal(659, 5, 20),   # Fifth (E)
])

print("A Major chord as a distribution")
ps.sonify(chord, duration=2.0).to_audio()

## Conclusion

Sonification provides an intuitive way to understand probability distributions:

- **Tight distributions** (low variance) sound focused and clear
- **Wide distributions** (high variance) sound diffuse and shimmery  
- **Skewed distributions** emphasize certain frequency ranges
- **Uniform distributions** sound chaotic, like noise
- **More samples** create richer, more complex textures

Try experimenting with your own parameters and distributions!